In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph ,END

In [3]:
###sTATE

class myState(TypedDict):
    topic: str
    python_result: str
    aws_result: str
    ml_result: str

# Node 1
def python_node(state) -> dict[str,str]:
    return {
        "python_result":f"Python roadmap for {state['topic']} is variable, loops, functions"
    }

# Node 2
def aws_node(state) -> dict[str,str]:
    return {
        "aws_result":f"AWS roadmap for {state['topic']} is EC2, S3, Docker"
    }    
     
# Node 3
def ml_node(state) -> dict[str,str]:
    return {
        "ml_result":f"ML roadmap for {state['topic']} is naive bayes, SVM, PCA"
    } 

### bUILD gRAPH

builder = StateGraph(myState)
builder.add_node("python_node", python_node)
builder.add_node("aws_node", aws_node)
builder.add_node("ml_node", ml_node)

# parellel execution from start
builder.set_entry_point("python_node")
builder.set_entry_point("aws_node")
builder.set_entry_point("ml_node")

builder.add_edge("python_node", END)
builder.add_edge("aws_node", END)
builder.add_edge("ml_node", END)

#compile graph

app = builder.compile()

## run graph
result = app.invoke({"topic":"data science"})
print(result)

{'topic': 'data science', 'python_result': 'Python roadmap for data science is variable, loops, functions', 'aws_result': 'AWS roadmap for data science is EC2, S3, Docker', 'ml_result': 'ML roadmap for data science is naive bayes, SVM, PCA'}


In [4]:
# Memory and checkpointing

# Memory -> means langgraph can remember previous conversation or previous state.

# Checkpointing -> means langgraph can save the state of the graph at a certain point and can resume from that point later.
                    # if graph stops in middle it can resume from the last checkpoint instead of starting from the beginning.


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph ,END,START
from langgraph.checkpoint.memory import MemorySaver

In [6]:
## state
class chatState(TypedDict):
    user_name : str
    question : str
    answer : str
    

In [16]:
###Node
def chatbot_node(state) -> dict[str,str]:
    if "name" in state['question'].lower():
        return {"answer":f"your name is {state['user_name']}!"}
    else:
        return {"answer":"I don't know the answer to that question."}
    
#Build Graph
builder = StateGraph(chatState)
builder.add_node("chatbot_node", chatbot_node)
builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

memory = MemorySaver()

app =builder.compile(checkpointer=memory)
config={
    "configurable":{
        "thread_id": "user101"
    }
}

#first call
result1 = app.invoke({"user_name":"Alice", "question":"What is my name?"}, config=config)
print(result1)


{'user_name': 'Alice', 'question': 'What is my name?', 'answer': 'your name is Alice!'}
